# 06 — Evaluation, SHAP, and driver analysis

**Workstream**: Eval + SHAP  ·  **Owner**: Bella (backup: Deepak)  ·  **Last touched**: 2026-06-15

**What this notebook does**

1. **Group-performance audit.** Does the served model perform comparably across
   `static_facility_type` and full `static_zip`? Basic in-scope fairness check
   (full disparate-impact audit is Phase 2). Run on the **right-truncation
   filtered** test split, matching the served eval basis.
2. **Per-restaurant SHAP top drivers.** For every restaurant, compute the 3-5
   features doing most of the work behind its predicted score, with plain-English
   labels (`src/foodsafety/explain/shap_drivers.py`). These feed the detail page.

> **Scoring / contract artifacts are NOT written here.** The served
> `scores.parquet` + `scores.json` are produced by the single writer,
> `scripts/retrain_baseline_sigmoid.py` (sigmoid-calibrated). This notebook is
> analysis only — see § 6.

**Production estimator**: calibrated baseline logistic regression (sigmoid,
served via the script). Feature set is whatever `baseline.py::ALL_FEATURES`
currently pins. Per CLAUDE.md's gate, XGBoost did not clear baseline on PR-AUC
AND precision@10%, so baseline ships. SHAP for logreg has a closed form
(coef × scaled feature value) so we compute it directly without the `shap`
package.

## 1. Setup

In [ ]:
import sys
import json
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parent
if str(_PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT / 'src'))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from foodsafety.config import MODELS_DIR, PROCESSED_DIR
from foodsafety.utils.time import temporal_split
from foodsafety.models.baseline import ALL_FEATURES, LABEL_COL
from foodsafety.models.evaluate import evaluate, decile_lift_table
from foodsafety.explain.shap_drivers import linear_contributions, top_drivers_for_row, FEATURE_LABELS
from foodsafety.serve.predict_batch import build_scores_table, write_scores_json, score_to_tier

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 2. Load the production model + features

Pick the most recent `baseline_*.joblib` from `data/models/`. The same
stamp file's `metadata.json` records the cutoffs we'll re-use.

In [ ]:
model_files = sorted(MODELS_DIR.glob('baseline_[0-9]*.joblib'))
if not model_files:
    raise SystemExit('No baseline model found. Run notebook 04 first.')

model_path = model_files[-1]
meta_path = model_path.with_name(model_path.stem + '_metadata.json')

model = joblib.load(model_path)
meta = json.loads(meta_path.read_text())

print(f'model:    {model_path.name}')
print(f'metadata: train_end={meta["split"]["train_end"]}, val_end={meta["split"]["val_end"]}')
print(f'features: {len(meta["features"]["all"])}')

features = pd.read_parquet(PROCESSED_DIR / 'features.parquet')
features['inspection_date'] = pd.to_datetime(features['inspection_date'])
for c in features.columns:
    if c.startswith('flag_kw_'):
        features[c] = features[c].astype('int8')
print(f'features: {len(features):,} rows × {features.shape[1]} cols')

## 3. Group-performance audit

Compute PR-AUC and precision@10% within each `static_facility_type` and each
full `static_zip` (5-digit). We deliberately do NOT use `static_zip3` — in
Chicago it's ~one bucket ("606"), so a zip-prefix audit checks nothing.

  - Per CLAUDE.md: "no group < 50% of overall PR-AUC" — a group with at least
    ~200 inspections should not collapse to <half the overall score. Smaller
    groups have noisy estimates; we report them but don't treat them as evidence.
  - This is the basic in-scope check; the full disparate-impact audit
    (statistical parity, equalised odds, demographic join) is Phase 2.

We use the same chronological test split as notebooks 04/05, **right-truncation
filtered** to match the served-model eval basis — otherwise the numbers aren't
comparable to the tracked served metrics.

In [ ]:
# RT-FILTER to match the served eval basis. Right-truncated rows (their 180-day
# forward window extends past the dataset max date) have UNDER-COUNTED labels, so
# the served script (retrain_baseline_sigmoid.py) drops them before splitting.
# Without this the audit + headline PR-AUC are on a ~2x-larger, label-biased test
# set and aren't comparable to the tracked served metrics.
features_eval = (
    features.loc[~features['right_truncated']].reset_index(drop=True)
    if 'right_truncated' in features.columns else features
)

split = temporal_split(
    features_eval,
    train_end=meta['split']['train_end'],
    val_end=meta['split']['val_end'],
)
test = split.test.copy()
test['risk_score'] = model.predict_proba(test[ALL_FEATURES])[:, 1]
test['y'] = test[LABEL_COL].astype(int)

from foodsafety.models.evaluate import precision_at_k, top_decile_lift
from sklearn.metrics import average_precision_score

def _group_metrics(g):
    if len(g) < 50:
        return pd.Series({'n': len(g), 'positive_rate': float('nan'),
                          'pr_auc': float('nan'), 'precision_at_10pct': float('nan')})
    return pd.Series({
        'n': len(g),
        'positive_rate': float(g['y'].mean()),
        'pr_auc': float(average_precision_score(g['y'], g['risk_score'])),
        'precision_at_10pct': precision_at_k(g['y'], g['risk_score'], 0.10),
    })

print(f'test rows (RT-filtered): {len(test):,}')
print('--- by static_facility_type ---')
by_facility = (
    test.groupby('static_facility_type', observed=True)
    .apply(_group_metrics, include_groups=False)
    .dropna()
    .sort_values('n', ascending=False)
    .round(3)
)
print(by_facility.to_string())

In [ ]:
# Geography audit on FULL static_zip, NOT static_zip3: in Chicago static_zip3 is
# ~one bucket ("606" = ~all rows), so a zip-PREFIX audit checks nothing. Full
# 5-digit ZIP is the meaningful geographic axis. (static_zip stays in the parquet
# for this audit even though it was dropped from the model feature set — DR 0004.)
print('--- by static_zip (top 15 by n) ---')
by_zip = (
    test.groupby('static_zip', observed=True)
    .apply(_group_metrics, include_groups=False)
    .dropna()
    .sort_values('n', ascending=False)
    .head(15)
    .round(3)
)
print(by_zip.to_string())

# Headline fairness check (on the RT-filtered test set, matching the served basis)
overall_pr_auc = float(average_precision_score(test['y'], test['risk_score']))
min_acceptable = overall_pr_auc * 0.5
print(f'\nOverall PR-AUC on test (RT-filtered): {overall_pr_auc:.3f}')
print(f'CLAUDE.md fairness threshold: group PR-AUC ≥ {min_acceptable:.3f}')
weak = by_facility[by_facility['pr_auc'] < min_acceptable]
if len(weak):
    print('\n⚠️  Facility types below threshold:')
    print(weak.to_string())
else:
    print('✓ All facility types with ≥50 test rows meet the threshold.')

## 4. Global feature impact (mean |contribution|)

Per-feature mean of `|log-odds contribution|` on the test set. This is the
global view that goes in the model card. The detail-page driver bars use
per-row contributions (computed in § 5).

In [ ]:
contrib_test = linear_contributions(model, test[ALL_FEATURES], original_features=ALL_FEATURES)

global_impact = contrib_test.abs().mean().sort_values(ascending=False)
print('Top 15 features by mean |log-odds contribution|:')
print(global_impact.head(15).round(4).to_string())

ax = global_impact.head(15)[::-1].plot.barh(color='#15110D', figsize=(8, 6))
ax.set_xlabel('Mean |log-odds contribution|')
ax.set_title('Production model — global feature impact (test set)')
plt.tight_layout(); plt.show()

## 5. Score every restaurant + build the contract artifact

`build_scores_table` produces one row per `license_id` anchored on the
restaurant's most recent inspection. It computes risk score, risk tier,
top-4 drivers (plain-English labels), and 90-day trend slope.

Output schema matches `docs/interface_contracts.md` § 3.

Wall-clock: ~30-60 s. Most of the time is the per-row SHAP attribution +
trend OLS fit; could be vectorised further if it grows.

In [ ]:
%%time
scores = build_scores_table(
    model=model,
    features=features,
    feature_columns=ALL_FEATURES,
    n_drivers=4,
)
print(f'\nscores: {len(scores):,} rows × {scores.shape[1]} cols')

In [ ]:
# Quick eyeball — top 10 by risk_score
preview = scores.sort_values('risk_score', ascending=False).head(10)
preview[['license_id', 'dba_name', 'address', 'risk_score', 'risk_tier', 'trend_slope_90d']].to_string()

In [ ]:
# Tier distribution
print('Tier distribution across all restaurants:')
print(scores['risk_tier'].value_counts().to_string())
print()

# Score distribution
print(scores['risk_score'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(3).to_string())

ax = scores['risk_score'].plot.hist(bins=40, color='#15110D', alpha=0.85,
                                     title='Risk-score distribution across restaurants')
ax.set_xlabel('Risk score'); plt.tight_layout(); plt.show()

### 5a. Sample driver lists

Spot-check three restaurants — one from each of High, Elevated, Low — and
print their top drivers. The labels are what end up on the detail page.

In [ ]:
for tier in ['High', 'Elevated', 'Low']:
    subset = scores[scores['risk_tier'] == tier]
    if subset.empty:
        continue
    sample = subset.iloc[0]
    print(f'--- {tier}: {sample["dba_name"]} ({sample["risk_score"]:.2f}) ---')
    for d in sample['top_drivers']:
        sign = '+' if d['shap'] > 0 else '−'
        print(f'  · {d["label"]:<55} ({sign}{abs(d["shap"]):.3f})')
    print()

## 6. Scoring — produced by the served script, NOT this notebook

The served `data/predictions/scores.parquet` and `app/public/data/scores.json`
are written by **`scripts/retrain_baseline_sigmoid.py`** — the *single writer*,
using the **sigmoid-calibrated** served model (decision records 0001 / 0002).

This notebook computes `scores` in-memory above only for SHAP / driver
inspection (§ 5) and **does not write the served artifacts**. Writing them here
from the *isotonic* baseline was a dual-writer that reintroduced the isotonic
score-tie problem the sigmoid switch fixed — removed per the methodology review.

In [ ]:
# The served scores.parquet is written by scripts/retrain_baseline_sigmoid.py
# (single writer, sigmoid). This notebook does NOT write it — we only inspect the
# in-memory `scores` table for SHAP / driver analysis.
print('tier distribution (in-memory `scores`, NOT written):')
print(scores['risk_tier'].value_counts().to_string())
print()
print(scores['risk_score'].describe(percentiles=[0.5, 0.9, 0.99]).round(3).to_string())

In [ ]:
# The served app/public/data/scores.json is produced by the sigmoid script,
# NOT here. (This cell previously wrote it from the ISOTONIC model — a dual-writer
# that reintroduced UI score-ties; removed per the methodology review / DR 0001.)
print('scores.json is produced by scripts/retrain_baseline_sigmoid.py (single writer, sigmoid).')

## 7. Hand-off

**Outputs**:
  - `data/predictions/scores.parquet` — Python pipeline artifact (cross-team contract)
  - `app/public/data/scores.json` — Next.js input. The web app **auto-drops
    the demo banner** when this file replaces the `scores_mock.json` fallback.

**Verification**: `cd app && pnpm dev` (or `npm run dev`) and visit
`http://localhost:3000` — the yellow demo banner should be gone, and the
real restaurant scores should appear.

**The walking-skeleton transition is complete.** Phase 1 mocked everything;
Phases 2-6 swapped each component for real implementations. The architecture
is the same; only the data changed.